In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("Housing Data(1).csv")

# Basic information
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 rows:")
print(df.head())

print("\nBasic Statistics:")
print(df.describe(include="all").T)

In [ ]:
missing = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_%": df.isnull().mean() * 100
})

missing = missing.sort_values("Missing_%", ascending=False)

print(missing)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

print(
    "Duplicate house IDs:",
    df["house_id"].duplicated().sum()
)

duplicate_ids = (
    df[df["house_id"].duplicated(keep=False)]
    .sort_values("house_id")
)

print(duplicate_ids)

In [ ]:
categorical_columns = [
    "house_type",
    "sales_type",
    "region",
    "area",
    "city"
]

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))

In [ ]:
numeric_columns = [
    "year_build",
    "purchase_price",
    "%_change_between_offer_and_purchase",
    "no_rooms",
    "sqm",
    "sqm_price",
    "nom_interest_rate%",
    "dk_ann_infl_rate%",
    "yield_on_mortgage_credit_bonds%"
]

print(df[numeric_columns].describe().T)

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f"Distribution of {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in numeric_columns:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Outliers - {col}")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in [
    "purchase_price",
    "sqm",
    "sqm_price",
    "no_rooms",
    "year_build"
]:
    print(f"\n--- {col} ---")
    print(df[col].nlargest(10).to_list())
    print(df[col].nsmallest(10).to_list())

In [ ]:
corr_columns = [
    "purchase_price",
    "sqm_price",
    "sqm",
    "no_rooms",
    "year_build",
    "nom_interest_rate%",
    "dk_ann_infl_rate%",
    "yield_on_mortgage_credit_bonds%",
    "%_change_between_offer_and_purchase"
]

corr = df[corr_columns].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
for col in ["purchase_price", "sqm", "sqm_price", "no_rooms"]:
    plt.figure(figsize=(9, 5))

    sns.boxplot(
        data=df,
        x="house_type",
        y=col
    )

    plt.title(f"{col} by House Type")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
for col in ["purchase_price", "sqm_price", "sqm"]:
    plt.figure(figsize=(9, 5))

    sns.boxplot(
        data=df,
        x="region",
        y=col
    )

    plt.title(f"{col} by Region")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
df["date"] = pd.to_datetime(df["date"])

print("Date range:")
print(df["date"].min())
print(df["date"].max())

In [ ]:
monthly_sales = (
    df.groupby(df["date"].dt.to_period("M"))
      .size()
)

monthly_sales.plot(figsize=(12, 5))

plt.title("Number of Sales Over Time")
plt.xlabel("Month")
plt.ylabel("Sales Count")
plt.tight_layout()
plt.show()

In [ ]:
analysis_df = df.copy()

# Date features
analysis_df["year"] = analysis_df["date"].dt.year
analysis_df["month"] = analysis_df["date"].dt.month
analysis_df["month_name"] = analysis_df["date"].dt.month_name()
analysis_df["quarter_num"] = analysis_df["date"].dt.quarter
analysis_df["year_month"] = analysis_df["date"].dt.to_period("M").astype(str)

# Property age
analysis_df["property_age"] = (
    analysis_df["year"] - analysis_df["year_build"]
)

# Price per room
analysis_df["price_per_room"] = (
    analysis_df["purchase_price"] /
    analysis_df["no_rooms"].replace(0, np.nan)
)

# Price per sqm - independently calculated
analysis_df["calculated_sqm_price"] = (
    analysis_df["purchase_price"] /
    analysis_df["sqm"].replace(0, np.nan)
)

# Difference between existing and calculated sqm price
analysis_df["sqm_price_difference"] = (
    analysis_df["sqm_price"] -
    analysis_df["calculated_sqm_price"]
)

# Implied offer price
analysis_df["offer_price"] = (
    analysis_df["purchase_price"] /
    (1 + analysis_df["%_change_between_offer_and_purchase"] / 100)
)

In [ ]:
print(
    analysis_df[
        [
            "purchase_price",
            "sqm",
            "sqm_price",
            "calculated_sqm_price",
            "sqm_price_difference"
        ]
    ].head(10)
)

In [ ]:
feature_columns = [
    "property_age",
    "price_per_room",
    "calculated_sqm_price",
    "sqm_price_difference",
    "offer_price"
]

print(analysis_df[feature_columns].describe().T)

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=analysis_df.sample(min(10000, len(analysis_df))),
    x="sqm",
    y="purchase_price",
    hue="house_type"
)

plt.title("Purchase Price vs SQM")
plt.tight_layout()
plt.show()

In [ ]:
features_to_check = [
    "purchase_price",
    "sqm_price",
    "calculated_sqm_price",
    "property_age",
    "price_per_room",
    "offer_price"
]

for col in features_to_check:

    Q1 = analysis_df[col].quantile(0.25)
    Q3 = analysis_df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = analysis_df[
        (analysis_df[col] < lower) |
        (analysis_df[col] > upper)
    ]

    print(
        f"{col}: {len(outliers)} outliers "
        f"({len(outliers) / len(analysis_df) * 100:.2f}%)"
    )

In [ ]:
feature_corr = analysis_df[
    [
        "purchase_price",
        "sqm",
        "no_rooms",
        "year_build",
        "property_age",
        "sqm_price",
        "price_per_room",
        "nom_interest_rate%",
        "dk_ann_infl_rate%",
        "yield_on_mortgage_credit_bonds%"
    ]
].corr()

plt.figure(figsize=(10, 7))

sns.heatmap(
    feature_corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Feature Correlation")
plt.tight_layout()
plt.show()

In [ ]:
feature_summary = (
    analysis_df
    .groupby("house_type")
    [
        [
            "property_age",
            "sqm",
            "no_rooms",
            "purchase_price",
            "sqm_price"
        ]
    ]
    .median()
)

print(feature_summary)

In [ ]:
region_summary = (
    analysis_df
    .groupby("region")
    [
        [
            "purchase_price",
            "sqm_price",
            "sqm",
            "no_rooms",
            "property_age"
        ]
    ]
    .median()
)

print(region_summary)